## Аггрегация новых признаков

In [1]:
import pandas as pd
import numpy as np
from math import sin, cos, sqrt, atan2, radians
from tqdm import tqdm
import os
import random
import asyncio
from typing import List, Dict, Any, Optional, Tuple

import aiohttp
from geopy.distance import geodesic


In [2]:
data = 'https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv'
df_train_full = pd.read_csv(data)

In [31]:
df_train_full.shape

(6200, 48)

In [47]:
df_train_full.head()

,id,atm_group,address_raw,address_geocoded,geo_lon,geo_lat,country,region,municipality,city,street,house
0,5.0,496.5,BUDENNOGO 7A ELISTA,"Россия, Республика Калмыкия, Элиста, улица С.М...",44.260605,46.318231,Россия,Республика Калмыкия,городской округ Элиста,Элиста,улица С.М. Будённого,7А
1,6.0,496.5,"HO CHI MIHN AVE, 19 ULYANOVSK","Россия, Ульяновск, проспект Хо Ши Мина, 19",48.300652,54.270443,Россия,Ульяновская область,городской округ Ульяновск,Ульяновск,проспект Хо Ши Мина,19
2,7.0,496.5,SHELESTA 116A KHABAROVSK,"Россия, Хабаровск, улица Шелеста, 116А",135.052594,48.520497,Россия,Хабаровский край,городской округ Хабаровск,Хабаровск,улица Шелеста,116А
3,8.0,496.5,ORDZHONIKIDZE 52 YAKUTSK,"Россия, Республика Саха (Якутия), Якутск, улиц...",129.721308,62.025566,Россия,Республика Саха (Якутия),городской округ Якутск,Якутск,улица Орджоникидзе,52
4,10.0,496.5,"VETERANOV AVE, 3 KRASNOKAMENS","Россия, Забайкальский край, Краснокаменск, про...",118.027480,50.090714,Россия,Забайкальский край,Краснокаменский муниципальный округ,Краснокаменск,проспект Ветеранов,3


In [33]:
geo_data_columns = ['id', 'atm_group', 'address_raw', 'address_geocoded', 'geo_lon', 'geo_lat', 'country', 'region', 'municipality', 'city', 'street', 'house']

In [34]:
df_train_full = df_train_full[geo_data_columns]

In [35]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
SEARCH_RADIUS = 1000
RADII = [100, 300, 500, 1000]
SLEEP = 2.0
SAVE_EVERY = 1
CONCURRENCY = 6               # аккуратно: Overpass не любит сильный параллелизм
TIMEOUT_S = 60
MAX_RETRIES = 4

CITY_SEARCH_RADIUS = 50_000
CITY_CENTER_THRESHOLD_M = 1500   # <-- порог для is_city_center

LANDUSE_LOOKUP_RADIUS = 500      # <-- радиус определения land_use_type

OUT_FILE = "new_poi_missed.csv"

## Сбор признаков через OSM

В этом ноутбуке признаки для банкоматов собираются из OpenStreetMap через API Overpass.
Для каждой точки (`geo_lat`, `geo_lon`) выполняются запросы по заранее заданным категориям объектов и формируются пространственные признаки.

### Что собираем
- **Плотность окружения**: количество объектов в радиусах `100`, `300`, `500`, `1000` м (`count_<category>_<R>m`).
- **Ближайший объект**: название и расстояние до ближайшего объекта в категории (`nearest_<category>_name`, `nearest_<category>_dist_m`).
- **Центр города**: расстояние до ближайшего `place=city` и бинарный признак `is_city_center` по порогу `CITY_CENTER_THRESHOLD_M`.
- **Тип землепользования**: `land_use_type` и `land_use_dist_m` по ближайшему `landuse` (или прокси через `building/office`, если `landuse` не найден).

### Как это считается
1. Для каждой категории строится Overpass-запрос по фильтрам из `OSM_FILTERS`.
2. Ответы OSM приводятся к точкам (`lat/lon` или `center`).
3. Геодезическое расстояние считается в метрах (`geopy.distance.geodesic`).
4. По расстояниям считаются агрегаты (counts по радиусам и nearest-признаки).

### Надёжность запросов
- Асинхронная обработка с ограничением параллелизма (`CONCURRENCY`).
- Повторы запросов при временных ошибках (`429/5xx`, timeout) с backoff.
- Промежуточное сохранение результата в CSV, чтобы можно было продолжить сбор после остановки.

### Результат
На выходе формируется таблица признаков по каждому ATM, пригодная для последующего анализа и обучения моделей.

In [ ]:
# OSM filters
OSM_FILTERS: Dict[str, Dict[str, Any]] = {
    "offices": {"filters": ['office=*', 'building=office'], "around": SEARCH_RADIUS},
    "payment_terminals": {"filters": ['amenity=payment_terminal'], "around": SEARCH_RADIUS},
    "money_transfer": {"filters": ['amenity=money_transfer'], "around": SEARCH_RADIUS},
    "shops_food_small": {"filters": ['shop=convenience', 'shop=discount'], "around": SEARCH_RADIUS},
    "hypermarkets": {"filters": ['shop=hypermarket'], "around": SEARCH_RADIUS},
    "markets": {"filters": ['amenity=marketplace', 'shop=market'], "around": SEARCH_RADIUS},
    "fitness_sport": {"filters": ['leisure=fitness_centre', 'leisure=sports_centre'], "around": SEARCH_RADIUS},
    "hotels_hostels": {"filters": ['tourism=hotel', 'tourism=hostel'], "around": SEARCH_RADIUS},

    "railway_stations": {
        "filters": [
            'railway=station',
            'railway=station&station=subway',
            'railway=station&subway=yes',
        ],
        "around": SEARCH_RADIUS,
    },

    "residential_buildings": {"filters": ['building=residential'], "around": SEARCH_RADIUS},
    "residential_landuse": {"filters": ['landuse=residential'], "around": SEARCH_RADIUS},
    "fuel": {"filters": ['amenity=fuel'], "around": SEARCH_RADIUS},
    "highway_pedestrian": {"filters": ['highway=pedestrian'], "around": SEARCH_RADIUS},

    "footway_100m": {"filters": ['highway=footway'], "around": 100, "override_radii": [100]},

    "landuse_mix": {"filters": ['landuse~"commercial|retail|industrial|residential"'], "around": SEARCH_RADIUS},
}

OSM_NEAREST: Dict[str, Dict[str, Any]] = {
    "place_city": {"filters": ['place=city'], "around": CITY_SEARCH_RADIUS}
}


# Вспомогательные функции
def geodist_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    return float(geodesic((lat1, lon1), (lat2, lon2)).meters)


def normalize_filter(f: str) -> str:
    f = f.strip()
    if "&" in f:
        return f
    if '="' in f or '~"' in f:
        return f
    if "=" in f:
        k, v = f.split("=", 1)
        k, v = k.strip(), v.strip()
        if v == "*" or v == "":
            return k  # presence
        return f'{k}="{v}"'
    return f


def build_overpass_query(lat: float, lon: float, filters: List[str], around_m: int) -> str:
    parts = []
    for raw in filters:
        raw = raw.strip()
        if not raw:
            continue

        if "&" in raw:
            chunks = [normalize_filter(x) for x in raw.split("&") if x.strip()]
            cond = "".join([f'[{c}]' for c in chunks])
            parts.append(f"node{cond}(around:{around_m},{lat},{lon});")
            parts.append(f"way{cond}(around:{around_m},{lat},{lon});")
            parts.append(f"relation{cond}(around:{around_m},{lat},{lon});")
            continue

        cond = normalize_filter(raw)
        parts.append(f"node[{cond}](around:{around_m},{lat},{lon});")
        parts.append(f"way[{cond}](around:{around_m},{lat},{lon});")
        parts.append(f"relation[{cond}](around:{around_m},{lat},{lon});")

    query = f"""
[out:json][timeout:60];
(
{chr(10).join(parts)}
);
out center;
"""
    return query


def extract_point(el: Dict[str, Any]) -> Optional[Tuple[float, float]]:
    el_lat = el.get("lat") or (el.get("center") or {}).get("lat")
    el_lon = el.get("lon") or (el.get("center") or {}).get("lon")
    if el_lat is None or el_lon is None:
        return None
    return float(el_lat), float(el_lon)


async def fetch_overpass(session: aiohttp.ClientSession, query: str, sem: asyncio.Semaphore) -> List[Dict[str, Any]]:
    backoff = 2.0
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with sem:
                async with session.get(OVERPASS_URL, params={"data": query}, timeout=TIMEOUT_S) as resp:
                    status = resp.status
                    if status == 200:
                        data = await resp.json(content_type=None)
                        return data.get("elements", []) or []

                    if status in (429, 504, 502, 503, 500):
                        await asyncio.sleep(backoff + random.uniform(0.0, 0.7))
                        backoff *= 1.8
                        continue

                    text = await resp.text()
                    print(f"Ошибка Overpass {status}: {text[:200]}")
                    return []
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            if attempt == MAX_RETRIES:
                print(f"Ошибка запроса (последняя попытка): {e}")
                return []
            await asyncio.sleep(backoff + random.uniform(0.0, 0.7))
            backoff *= 1.8

    return []


async def analyze_category(
    session: aiohttp.ClientSession,
    sem: asyncio.Semaphore,
    lat: float,
    lon: float,
    key: str,
    cfg: Dict[str, Any],
    radii_default: List[int],
) -> Dict[str, Any]:
    around_m = int(cfg.get("around", SEARCH_RADIUS))
    filters = cfg.get("filters", [])
    radii = cfg.get("override_radii", radii_default)

    query = build_overpass_query(lat, lon, filters, around_m)
    elements = await fetch_overpass(session, query, sem)

    out: Dict[str, Any] = {}
    if not elements:
        for R in radii:
            out[f"count_{key}_{R}m"] = 0
        out[f"nearest_{key}_name"] = ""
        out[f"nearest_{key}_dist_m"] = None
        return out

    dists: List[Tuple[str, float]] = []
    for el in elements:
        pt = extract_point(el)
        if not pt:
            continue
        el_lat, el_lon = pt
        tags = el.get("tags") or {}
        name = tags.get("name", "") or ""
        dist = geodist_m(lat, lon, el_lat, el_lon)
        dists.append((name, dist))

    if not dists:
        for R in radii:
            out[f"count_{key}_{R}m"] = 0
        out[f"nearest_{key}_name"] = ""
        out[f"nearest_{key}_dist_m"] = None
        return out

    nearest_name, nearest_dist = min(dists, key=lambda x: x[1])
    out[f"nearest_{key}_name"] = nearest_name
    out[f"nearest_{key}_dist_m"] = round(nearest_dist, 1)

    for R in radii:
        out[f"count_{key}_{R}m"] = sum(1 for _, d in dists if d <= R)

    return out


async def analyze_nearest_city_center(
    session: aiohttp.ClientSession,
    sem: asyncio.Semaphore,
    lat: float,
    lon: float,
) -> Dict[str, Any]:
    cfg = OSM_NEAREST["place_city"]
    query = build_overpass_query(lat, lon, cfg["filters"], int(cfg["around"]))
    elements = await fetch_overpass(session, query, sem)

    out = {
        "city_center_name": "",
        "city_center_dist_m": None,
        "is_city_center": 0,
    }

    if not elements:
        return out

    best: Optional[Tuple[str, float]] = None
    for el in elements:
        pt = extract_point(el)
        if not pt:
            continue
        el_lat, el_lon = pt
        tags = el.get("tags") or {}
        name = tags.get("name", "") or tags.get("name:en", "") or ""
        dist = geodist_m(lat, lon, el_lat, el_lon)
        cand = (name, dist)
        if best is None or cand[1] < best[1]:
            best = cand

    if best:
        out["city_center_name"] = best[0]
        out["city_center_dist_m"] = round(best[1], 1)
        out["is_city_center"] = 1 if best[1] <= CITY_CENTER_THRESHOLD_M else 0

    return out


async def analyze_land_use_type(
    session: aiohttp.ClientSession,
    sem: asyncio.Semaphore,
    lat: float,
    lon: float,
) -> Dict[str, Any]:
    """
    Определяем land_use_type по ближайшему:
      1) landuse in {commercial, retail, industrial, residential}
      2) если не найден — по прокси: building=residential -> residential, office=* / building=office -> commercial
    Возвращаем: {"land_use_type": "...", "land_use_dist_m": ...}
    """
    # Сначала пробуем landuse=...
    landuse_filters = ['landuse~"commercial|retail|industrial|residential"']
    q1 = build_overpass_query(lat, lon, landuse_filters, LANDUSE_LOOKUP_RADIUS)
    els1 = await fetch_overpass(session, q1, sem)

    best_lu: Optional[Tuple[str, float]] = None
    for el in els1:
        pt = extract_point(el)
        if not pt:
            continue
        tags = el.get("tags") or {}
        lu = tags.get("landuse")
        if not lu:
            continue
        el_lat, el_lon = pt
        dist = geodist_m(lat, lon, el_lat, el_lon)
        cand = (lu, dist)
        if best_lu is None or cand[1] < best_lu[1]:
            best_lu = cand

    if best_lu:
        lu_raw, dist = best_lu
        # нормализуем в “типы” (можно расширить)
        if lu_raw in ("commercial", "retail"):
            lu_type = "commercial"
        elif lu_raw == "industrial":
            lu_type = "industrial"
        elif lu_raw == "residential":
            lu_type = "residential"
        else:
            lu_type = lu_raw

        return {"land_use_type": lu_type, "land_use_dist_m": round(dist, 1)}

    # Если landuse не нашли — прокси по building/office
    proxy_filters = ['building=residential', 'office=*', 'building=office']
    q2 = build_overpass_query(lat, lon, proxy_filters, LANDUSE_LOOKUP_RADIUS)
    els2 = await fetch_overpass(session, q2, sem)

    best_proxy: Optional[Tuple[str, float]] = None
    for el in els2:
        pt = extract_point(el)
        if not pt:
            continue
        tags = el.get("tags") or {}
        el_lat, el_lon = pt
        dist = geodist_m(lat, lon, el_lat, el_lon)

        if tags.get("building") == "residential":
            kind = "residential"
        elif tags.get("building") == "office" or ("office" in tags):
            kind = "commercial"
        else:
            continue

        cand = (kind, dist)
        if best_proxy is None or cand[1] < best_proxy[1]:
            best_proxy = cand

    if best_proxy:
        kind, dist = best_proxy
        return {"land_use_type": kind, "land_use_dist_m": round(dist, 1)}

    return {"land_use_type": "", "land_use_dist_m": None}


# Main
async def process_dataset_async(df: pd.DataFrame, dataset_name: str, out_file: str, start_from: int = 0) -> None:
    if os.path.exists(out_file):
        processed_df = pd.read_csv(out_file)
        processed_ids = set(processed_df["atm_id"])
        print(f"Найден файл {out_file}, продолжаем с {len(processed_ids)} записей")
    else:
        processed_df = pd.DataFrame()
        processed_ids = set()
        print(f"Файл {out_file} не найден, начинаем с нуля")

    results: List[Dict[str, Any]] = []
    start_index = max(len(processed_ids), start_from)

    sem = asyncio.Semaphore(CONCURRENCY)
    timeout = aiohttp.ClientTimeout(total=TIMEOUT_S + 10)
    headers = {"User-Agent": "atm-osm-analyzer/1.0"}

    async with aiohttp.ClientSession(timeout=timeout, headers=headers) as session:
        for i, row in df.iloc[start_index:].iterrows():
            lat, lon = float(row["geo_lat"]), float(row["geo_lon"])
            atm_id = row.get("id", i)

            print(f"\n[{dataset_name.upper()}] Банкомат {i+1}/{len(df)}: ({lat:.5f}, {lon:.5f})")

            row_result: Dict[str, Any] = {
                "atm_id": atm_id,
                "atm_lat": lat,
                "atm_lon": lon,
            }

            # --- параллельные задачи ---
            tasks = [
                analyze_category(session, sem, lat, lon, key, cfg, RADII)
                for key, cfg in OSM_FILTERS.items()
            ]

            # новые колонки
            tasks.append(analyze_land_use_type(session, sem, lat, lon))
            tasks.append(analyze_nearest_city_center(session, sem, lat, lon))

            chunks = await asyncio.gather(*tasks)

            for chunk in chunks:
                row_result.update(chunk)

            results.append(row_result)

            # мягкий троттлинг
            await asyncio.sleep(0.2 + random.uniform(0.0, 0.25))

            # Автосохранение
            if len(results) % SAVE_EVERY == 0:
                temp_df = pd.DataFrame(results)
                combined_df = pd.concat([processed_df, temp_df], ignore_index=True)
                combined_df.to_csv(out_file, index=False, encoding="utf-8-sig")
                print(f"💾 Промежуточное сохранение: {len(combined_df)} строк")
                processed_df = combined_df.copy()
                results = []

    final_df = pd.concat([processed_df, pd.DataFrame(results)], ignore_index=True)
    final_df.to_csv(out_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ Готово: обработано {len(final_df)} банкоматов ({dataset_name})")
    print(f"Файл сохранён: {out_file}")


In [ ]:
await process_dataset_async(df_train_full, "train", OUT_FILE)

## Описание признаков

In [ ]:
merged_df = pd.read_csv("merged_data.csv")

In [4]:
# Строки из df_train_full, которых нет в merged_df по atm_id
df_train_cmp = df_train_full.copy()

if "atm_id" not in df_train_cmp.columns:
    if "id" in df_train_cmp.columns:
        df_train_cmp["atm_id"] = df_train_cmp["id"]
    else:
        raise ValueError("В df_train_full нет столбца 'atm_id' или 'id'")

if "atm_id" not in merged_df.columns:
    raise ValueError("В merged_df нет столбца 'atm_id'")

missing_in_merged_df = (
    df_train_cmp.loc[~df_train_cmp["atm_id"].isin(merged_df["atm_id"])].copy()
    .sort_values("atm_id", kind="stable")
    .reset_index(drop=True)
)

print(f"Всего строк в df_train_full: {len(df_train_cmp):,}".replace(",", " "))
print(f"Нет в merged_df по atm_id: {len(missing_in_merged_df):,}".replace(",", " "))

missing_in_merged_df.head()

Всего строк в df_train_full: 6 200
Нет в merged_df по atm_id: 0


,id,atm_group,address_raw,address_geocoded,geo_lon,geo_lat,country,region,municipality,city,...,count_public_transport_300m,nearest_parking_dist_m,count_parking_300m,nearest_education_dist_m,count_education_300m,nearest_subway_dist_m,nearest_post_offices_dist_m,count_post_offices_300m,has_subway_nearby,atm_id


In [5]:
# колонки с названиями объектов
cols_to_del = [
    "nearest_offices_name", 
    "nearest_payment_terminals_name",
    "nearest_shops_food_small_name",
    "nearest_money_transfer_name",
    "nearest_payment_terminals_name",
    "nearest_hypermarkets_name",
    "nearest_fitness_sport_name",
    "nearest_hotels_hostels_name",
    "nearest_railway_stations_name",
    "nearest_residential_buildings_name",
    "nearest_residential_landuse_name",
    "nearest_fuel_name",
    "nearest_highway_pedestrian_name",
    "nearest_footway_100m_name"
    ]

merged_df = merged_df.drop(columns=cols_to_del, axis=1)

In [6]:
merged_df.to_csv("new_poi_merged_final.csv", index=False, encoding="utf-8-sig")

### Описание признаков `merged_df`

`merged_df` — таблица с геопризнаками окружения банкоматов (по данным OSM).

**Базовые колонки**
- `atm_id` — идентификатор банкомата.
- `atm_lat`, `atm_lon` — широта и долгота банкомата.

**Шаблоны признаков**
- `nearest_<категория>_name` — название ближайшего объекта категории.
- `nearest_<категория>_dist_m` — расстояние до ближайшего объекта категории, в метрах.
- `count_<категория>_100m`, `count_<категория>_300m`, `count_<категория>_500m`, `count_<категория>_1000m` — количество объектов категории в радиусах 100/300/500/1000 м.

**Категории, для которых рассчитаны признаки**
- `offices` — офисы.
- `payment_terminals` — платёжные терминалы.
- `money_transfer` — точки денежных переводов.
- `shops_food_small` — небольшие продуктовые магазины.
- `hypermarkets` — гипермаркеты.
- `markets` — рынки.
- `fitness_sport` — фитнес/спорт-объекты.
- `hotels_hostels` — отели и хостелы.
- `railway_stations` — ж/д станции.
- `residential_buildings` — жилые здания.
- `residential_landuse` — зоны жилой застройки.
- `fuel` — АЗС.
- `highway_pedestrian` — пешеходная инфраструктура (тип highway=pedestrian).
- `landuse_mix` — смешанная застройка/землепользование.

**Специальные колонки**
- `nearest_footway_100m_name` — название ближайшего пешеходного пути (footway) в контексте зоны 100 м.
- `nearest_footway_100m_dist_m` — расстояние до ближайшего footway, м.
- `count_footway_100m_100m` — количество footway-объектов в радиусе 100 м.
- `land_use_type` — тип ближайшего землепользования.
- `land_use_dist_m` — расстояние до ближайшего полигона `land_use_type`, м.
- `city_center_name` — название города/центра, относительно которого считалась дистанция до центра.
- `city_center_dist_m` — расстояние до центра города, м.
- `is_city_center` — бинарный признак центра (`1` — относится к центру, `0` — нет).

> Если рядом нет объектов нужной категории, поля `nearest_*_name` и/или `nearest_*_dist_m` могут быть пустыми (`NaN`).